# Interoperability

No GPU programming approach covers every computational pattern equally well.
A common situation in practice: an application is written in one approach, e.g. OpenMP or Kokkos, and a single *hotspot* cannot be expressed efficiently, or at all.
Instead of porting the whole application, only the hotspot is replaced by a hand-written CUDA kernel (or HIP on AMD GPUs).

This works because, on NVIDIA GPUs, all approaches discussed in this workshop are built on top of CUDA.
They use the same device memory, so data can be shared by passing *raw device pointers*.

## Ingredients

Calling a CUDA kernel from an application written in approach X requires three ingredients:
1. **Data:** obtain the raw device pointer of data managed by X
2. **Synchronization:** make sure that X has finished working on the data before CUDA takes over, and vice versa
3. **Toolchain:** compile the CUDA part with `nvcc` and link it with the rest of the application

## Example

We once more use the increase example.
To mimic a hotspot implemented in CUDA, even iterations are executed with approach X and odd iterations with a CUDA kernel.
Since each iteration increases all elements by one, the final result, and thus the check, is the same as before.

```c++
for (size_t i = 0; i < nIt; ++i) {
    if (0 == i % 2)
        increase(data, nx);         // approach X
    else
        increaseInterop(data, nx);  // CUDA
}
```

The CUDA part is implemented in [increase-cuda.cu](../src/interop/increase-cuda.cu).
It consists of a kernel and a host function that launches it.

```c++
__global__ void increaseKernel(double *data, size_t nx) {
    const size_t i0 = blockIdx.x * blockDim.x + threadIdx.x;

    if (i0 < nx) {
        data[i0] += 1;
    }
}

void increaseCuda(double *d_data, size_t nx) {
    const size_t blockSize = 256;
    const size_t numBlocks = (nx + blockSize - 1) / blockSize;

    increaseKernel<<<numBlocks, blockSize>>>(d_data, nx);

    checkCudaError(cudaDeviceSynchronize(), true);
}
```

Two design decisions keep the CUDA part usable from all approaches:
* The header [increase-cuda.h](../src/interop/increase-cuda.h) only declares `increaseCuda` and contains no CUDA types or syntax.
  It can thus be included by any compiler, and the rest of the application does not need to know about CUDA.
* `increaseCuda` waits for the kernel to finish before returning.
  This is simple and safe, since other approaches may use their own CUDA streams.
  The price is that the CUDA kernel cannot overlap with other work.

The CUDA part is compiled once into an object file with `nvcc -c`.
It is then linked with each of the following approaches.

In [ ]:
!nvcc -O3 -std=c++17 -arch=sm_86 -c -o ../build/interop/increase-cuda.o ../src/interop/increase-cuda.cu

## OpenMP

**Data:** within a `target data` construct, the `use_device_addr` clause replaces host addresses of mapped data with the corresponding device addresses.

```c++
inline void increaseInterop(double *data, size_t nx) {
#pragma omp target data use_device_addr(data)
    {
        increaseCuda(data, nx);
    }
}
```

Alternatively, `omp_get_mapped_ptr(data, omp_get_default_device())` returns the device address.
The older `use_device_ptr` clause also works but is flagged as deprecated by `nvc++`.

**Synchronization:** target regions synchronize implicitly at their end.

**Toolchain:** `nvc++` links the CUDA runtime when `-cuda` is passed.

The complete example code is available in [increase-interop-omp-target.cpp](../src/interop/increase-interop-omp-target.cpp).

In [ ]:
!nvc++ -O3 -std=c++17 -mp=gpu -target=gpu -cuda -o ../build/interop/increase-interop-omp-target ../src/interop/increase-interop-omp-target.cpp ../build/interop/increase-cuda.o

In [ ]:
!../build/interop/increase-interop-omp-target

## OpenACC

**Data:** within a `host_data` construct, the `use_device` clause replaces host addresses with the corresponding device addresses.

```c++
inline void increaseInterop(double *data, size_t nx) {
#pragma acc host_data use_device(data)
    {
        increaseCuda(data, nx);
    }
}
```

Alternatively, `acc_deviceptr(data)` returns the device address.

**Synchronization:** compute constructs without `async` synchronize implicitly at their end.

**Toolchain:** as with OpenMP, pass `-cuda` to `nvc++`.

The complete example code is available in [increase-interop-openacc.cpp](../src/interop/increase-interop-openacc.cpp).

In [ ]:
!nvc++ -O3 -std=c++17 -acc=gpu -target=gpu -cuda -o ../build/interop/increase-interop-openacc ../src/interop/increase-interop-openacc.cpp ../build/interop/increase-cuda.o

In [ ]:
!../build/interop/increase-interop-openacc

## Modern C++

**Data:** with `-stdpar=gpu`, heap memory is managed memory (or plain system memory on systems with heterogeneous memory management).
It is directly accessible in CUDA kernels, so the pointer can be passed as is.

```c++
inline void increaseInterop(double *data, size_t nx) {
    increaseCuda(data, nx);
}
```

**Synchronization:** parallel algorithms are synchronous with respect to the CPU.

**Toolchain:** pass `-cuda` to `nvc++`.

The complete example code is available in [increase-interop-std-par.cpp](../src/interop/increase-interop-std-par.cpp).

In [ ]:
!nvc++ -O3 -std=c++17 -stdpar=gpu -target=gpu -gpu=cc86 -cuda -o ../build/interop/increase-interop-std-par ../src/interop/increase-interop-std-par.cpp ../build/interop/increase-cuda.o

In [ ]:
!../build/interop/increase-interop-std-par

## Thrust

**Data:** `thrust::raw_pointer_cast` yields the raw device pointer of a `device_vector`.

```c++
inline void increaseInterop(thrust::device_vector<double> &data, size_t nx) {
    increaseCuda(thrust::raw_pointer_cast(data.data()), nx);
}
```

**Synchronization:** Thrust algorithms are synchronous with respect to the CPU.

**Toolchain:** Thrust code is compiled with `nvcc` anyway.
The CUDA kernel could therefore also be placed in the same file.

The complete example code is available in [increase-interop-thrust.cu](../src/interop/increase-interop-thrust.cu).

In [ ]:
!nvcc -O3 -std=c++17 --extended-lambda -arch=sm_86 -o ../build/interop/increase-interop-thrust ../src/interop/increase-interop-thrust.cu ../build/interop/increase-cuda.o

In [ ]:
!../build/interop/increase-interop-thrust

## Kokkos

**Data:** with the CUDA backend, `data()` returns the raw device pointer of a `View`.

**Synchronization:** Kokkos kernels execute asynchronously.
A `Kokkos::fence()` is required before CUDA takes over.

```c++
inline void increaseInterop(Kokkos::View<double *> &data, size_t nx) {
    Kokkos::fence();

    increaseCuda(data.data(), nx);
}
```

**Toolchain:** link the object file when building with `nvcc_wrapper`.

The complete example code is available in [increase-interop-kokkos.cpp](../src/interop/increase-interop-kokkos.cpp).

In [ ]:
!/root/kokkos/install-cuda-86/bin/nvcc_wrapper -O3 -march=native -std=c++20 -arch=sm_86 --expt-extended-lambda --expt-relaxed-constexpr -I/root/kokkos/install-cuda-86/include -L/root/kokkos/install-cuda-86/lib -o ../build/interop/increase-interop-kokkos ../src/interop/increase-interop-kokkos.cpp ../build/interop/increase-cuda.o -lkokkoscore -ldl -lcuda

In [ ]:
!../build/interop/increase-interop-kokkos

## SYCL

**Data:** if the SYCL device uses the CUDA backend, device USM pointers are regular CUDA device pointers.
The backend can be checked at runtime.

```c++
if (sycl::backend::cuda != q.get_device().get_backend()) { /* ... */ }
```

**Synchronization:** SYCL kernels execute asynchronously.
A `q.wait()` is required before CUDA takes over - an in-order queue only orders work *within* the queue.

```c++
inline void increaseInterop(sycl::queue &q, double *data, size_t nx) {
    q.wait();

    increaseCuda(data, nx);
}
```

**Toolchain:** link the object file and the CUDA runtime when building with `acpp`.
Linking it statically keeps it independent of the CUDA version AdaptiveCpp was built with and requires no library path at runtime.

The complete example code is available in [increase-interop-sycl.cpp](../src/interop/increase-interop-sycl.cpp).

In [ ]:
!acpp -O3 -march=native -std=c++17 -o ../build/interop/increase-interop-sycl ../src/interop/increase-interop-sycl.cpp ../build/interop/increase-cuda.o -L/opt/nvidia/hpc_sdk/Linux_x86_64/26.1/cuda/lib64 -lcudart_static

In [ ]:
!../build/interop/increase-interop-sycl

## Pitfalls

* **Missing synchronization:** work submitted by different approaches is not ordered automatically.
  Every hand-over of data requires the producer to have finished.
* **Name clashes:** all parts end up in one executable.
  For example, naming the CUDA kernel `increase` clashes with the `increase` function of the OpenACC version: both have the same symbol name, and the kernel launch fails with `invalid argument`.
  Use unique names or namespaces.
* **Missing CUDA runtime:** without `-cuda` (`nvc++`) or `-lcudart`/`-lcudart_static` (other compilers), linking fails with undefined references to CUDA functions.
* **Portability:** a CUDA hotspot ties the application to NVIDIA GPUs.
  On AMD GPUs, the same pattern works with HIP.
  Keeping the original implementation of approach X as a fallback preserves portability.

## Next Step

Return to the [next steps](./next-steps.ipynb) notebook.